<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="assets/content/images/ensemble_thumbnail.png" align="center" width="20%">
</div>

<br>

# ENSEMBLE METHODS AND AUTOML

<br>

**About:** An introduction to ensemble learning theory followed by a hands-on walkthrough using two AutoML frameworks - LightAutoML and FlaML - to build and combine survival predictions on the Titanic dataset.

**Learning Goals:** After completing this notebook, you will be able to:
1. Explain why combining predictions from multiple diverse models reduces error
2. Distinguish bagging, boosting, stacking, and simple averaging as ensemble strategies
3. Use LightAutoML to train a time-budgeted binary classification pipeline
4. Use FlaML to train an automated classification model and generate probability predictions
5. Combine predictions from multiple models using simple averaging and interpret the effect of threshold selection

**Keywords:** ensemble learning, AutoML, LightAutoML, FlaML, model averaging, bias-variance tradeoff, classification

**Prerequisite Knowledge:** (1) Binary classification concepts (accuracy, AUC, precision/recall), (2) Python and pandas basics, (3) `01_titanic_eda_and_feature_engineering.ipynb`

**Target User:** Learners who understand basic ML concepts and want to see how AutoML tools can be combined into a simple, competitive ensemble

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 1: ENSEMBLE LEARNING - CONCEPTS](#Part_1)
> #### [PART 2: AUTOML WITH LIGHTAUTOML](#Part_2)
> #### [PART 3: AUTOML WITH FLAML](#Part_3)
> #### [PART 4: COMBINING PREDICTIONS AND THRESHOLD TUNING](#Part_4)

<br>

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **ENSEMBLE LEARNING** - CONCEPTS

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/ensemble_diagram.png" align="center" width="45%" padding="10"><br>
    <br>
</div>

#### CONTENTS:

> [PART 1.1: Why Combine Models?](#Part_1_1)<br>
> [PART 1.2: Types of Ensemble Methods](#Part_1_2)<br>
> [PART 1.3: The Role of Model Diversity](#Part_1_3)<br>

<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: WHY COMBINE MODELS?

<br>

## **Why** Ensemble?

A single machine learning model makes errors. Those errors come from two sources:

**Bias** - the model is too simple to capture the true relationship. A linear regression applied to a clearly non-linear problem will always be wrong in a systematic way regardless of how much data you add.

**Variance** - the model is too sensitive to the specific training sample. A deep decision tree trained on one random 80/20 split will produce a different tree on a different 80/20 split; both will be "right" for their training data but differ in their predictions for new inputs.

The **bias-variance tradeoff** means that reducing one source of error typically increases the other. Simple models have high bias but low variance; complex models have low bias but high variance. Ensembles are one way to escape this tradeoff:

- **Bagging** (Bootstrap Aggregating) reduces **variance** by training many high-variance models on different bootstrap samples and averaging their predictions. Random Forest is bagging applied to decision trees.
- **Boosting** reduces **bias** by training a sequence of weak models, each one correcting the errors of the previous. Gradient Boosted Trees (XGBoost, LightGBM, CatBoost) are the dominant boosting implementations.
- **Stacking** trains a meta-model whose inputs are the predictions of several base models - it learns when to trust each base model.
- **Simple averaging** takes the arithmetic mean of probability outputs from several independently trained models.

**Sources Consulted:**
- Bias-variance tradeoff: Hastie, T., Tibshirani, R., Friedman, J. (2009). *The Elements of Statistical Learning*, Chapter 7. [Online edition](https://hastie.su.domains/ElemStatLearn/)
- Ensemble methods overview: [scikit-learn Ensemble Guide](https://scikit-learn.org/stable/modules/ensemble.html)

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.2: TYPES OF ENSEMBLE METHODS

<br>

## **Types** of Ensembles

The table below organizes ensemble strategies by the mechanism they use to reduce error:

| Strategy | Mechanism | Key Example | Best For |
|---|---|---|---|
| Bagging | Independent models on bootstrap samples, aggregate by vote/average | Random Forest | High-variance models (deep trees) |
| Boosting | Sequential correction of residuals | XGBoost, LightGBM, CatBoost | High-bias models (shallow trees) |
| Stacking | Meta-model learns to weight base-model predictions | Level-2 logistic regression | When base models have complementary strengths |
| Simple averaging | Arithmetic mean of probability outputs | Mean of AutoML outputs | Fast, low-risk, often competitive with stacking |
| Voting (hard) | Majority vote across classifiers | VotingClassifier | When probability calibration is unreliable |

**AutoML as an ensemble tool**: Modern AutoML frameworks (H2O.ai, LightAutoML, FlaML) do not just select the best single model - they often build an ensemble internally. LightAutoML, for example, runs a blending step that mixes the outputs of multiple model families (linear, gradient-boosted trees, neural networks) weighted by out-of-fold validation performance. What we do in Part 4 - averaging across three AutoML outputs - is an outer ensemble on top of those inner ensembles.

___

**Note:** Stacking requires care around data leakage. The meta-model must be trained on out-of-fold predictions from the base models, not on predictions that saw the same training rows. LightAutoML's `fit_predict` returns out-of-fold predictions by default, making it compatible with stacking as a base model.

___

**Sources Consulted:**
- AutoML internal ensembling: [LightAutoML Documentation](https://lightautoml.readthedocs.io/en/latest/)
- Stacking and data leakage: [scikit-learn StackingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html)

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.3: THE ROLE OF MODEL DIVERSITY

<br>

## **Model Diversity** and Why It Matters

Combining models only helps when they disagree on which examples they get wrong. If all models make the same mistakes, averaging them cannot reduce error - you simply average the same wrong answer.

The formal statement: for an ensemble of $M$ models making independent, equally-capable predictions, the expected squared error of the average is $\frac{1}{M}$ of the expected squared error of any single model.

$$\text{Error}_{\text{ensemble}} = \frac{1}{M} \text{Error}_{\text{individual}}$$

where the $\frac{1}{M}$ factor holds exactly only when model errors are **uncorrelated**. In practice, models trained on the same dataset are never fully independent - so the gain is partial but still real.

This means the ideal ensemble is built from models that:
1. Are individually accurate (strong but not identical).
2. Are diverse in the types of errors they make.

___

**Note:** The correlation among predictions is not the same as the correlation among errors, but in practice highly correlated predictions usually imply highly correlated errors. In Part 4 we measure this directly.

___

**Sources Consulted:**
- Ensemble diversity: Breiman, L. (2001). Random Forests. *Machine Learning*, 45, 5-32. [PDF](https://link.springer.com/article/10.1023/A:1010933404324)
- Theoretical analysis of ensemble error: [Dietterich, T.G. (2000). Ensemble Methods in Machine Learning](https://link.springer.com/chapter/10.1007/3-540-45014-9_1)

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **A random forest makes predictions by averaging the outputs of 100 decision trees. Each tree was trained on a different bootstrap sample of the data. In 2-3 sentences: explain why this averaging tends to produce better predictions than any single tree, and name one condition under which adding more trees stops improving performance.**

<br>

In [ ]:
# This is a written response question - your answer in the markdown cell below


*ANSWER HERE*

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **AUTOML** WITH LIGHTAUTOML

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/lightautoml_logo.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

#### CONTENTS:

> [PART 2.1: Setup and Installation](#Part_2_1)<br>
> [PART 2.2: Training and Out-of-Fold Evaluation](#Part_2_2)<br>
> [PART 2.3: Generating Test Predictions](#Part_2_3)<br>

<a id='Part_2_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.1: SETUP AND INSTALLATION

<br>

**LightAutoML** (developed by Sber AI Lab) is an automated machine learning framework that searches over model families (linear, LightGBM, CatBoost, neural networks) within a configurable time budget. It handles feature preprocessing internally - including categorical encoding and missing value handling - but benefits from having already-clean numeric inputs.

The `TabularUtilizedAutoML` preset uses a blending strategy internally: it trains multiple models and learns weights for combining them based on out-of-fold validation scores. The time budget is split between model search and the blending step.

___

**Note:** LightAutoML requires PyTorch. On machines without a GPU the training still runs but will use CPU-only mode. Runtimes scale with the `timeout` parameter (seconds).

___

**Sources Consulted:**
- LightAutoML: [readthedocs](https://lightautoml.readthedocs.io/en/latest/) - retrieved 2026-08-25
- TabularUtilizedAutoML API: [sb-ai-lab/LightAutoML on GitHub](https://github.com/sb-ai-lab/LightAutoML) - retrieved 2026-08-25

In [ ]:
# Install LightAutoML (skip if already installed)
# TODO: verify package version against current PyPI before pinning
!pip install -U lightautoml -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
from sklearn.metrics import roc_auc_score

# LightAutoML imports
# TODO: verify import paths against current lightautoml docs (retrieved 2026-08-25)
from lightautoml.automl.presets.tabular_presets import TabularUtilizedAutoML
from lightautoml.tasks import Task

# Reproducibility and resource limits
N_THREADS   = 4    # CPU threads for LightGBM and linear models
RANDOM_STATE = 42
TIMEOUT     = 300  # seconds; increase for better search coverage on larger datasets

np.random.seed(RANDOM_STATE)
torch.set_num_threads(N_THREADS)
print(f"PyTorch version: {torch.__version__}")

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.2: TRAINING AND OUT-OF-FOLD EVALUATION

<br>

**Roles** tell LightAutoML which column is the prediction target and which columns to ignore (e.g., identifier columns). The feature matrix we built in notebook 01 has already dropped `PassengerId` from the training set, but the test set retains it. We specify it in `drop` so LightAutoML ignores it if present.

`fit_predict` trains the pipeline and returns **out-of-fold (OOF) predictions**: for each training row, the prediction was made by a model that did not see that row during training. This gives an unbiased estimate of generalization error without touching the test set.

The OOF AUC is the primary metric for comparing models in this notebook - it is computed entirely on training data, making it a fair comparison against FlaML's OOF score in Part 3.

In [ ]:
# Load the preprocessed data produced by notebook 01
# If you ran notebook 01 and saved the DataFrames, load them here.
# For convenience, we re-run the preprocessing inline.
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

# --- Minimal preprocessing (mirrors notebook 01) ---
combine = [train_df, test_df]

# Title extraction
for dataset in combine:
    dataset['Title'] = dataset['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    rare = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
    dataset['Title'] = dataset['Title'].replace(rare, 'Rare')
    dataset['Title'] = dataset['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    dummies = pd.get_dummies(dataset['Title'])
    dataset[dummies.columns] = dummies

train_df = train_df.drop(['Name', 'Title'], axis=1)
test_df  = test_df.drop(['Name', 'Title'], axis=1)
combine  = [train_df, test_df]

# Sex
for dataset in combine:
    dataset['Sex'] = dataset['Sex'].map({'female': 1, 'male': 0}).astype(int)

# Age imputation
guess_ages = np.zeros((2, 3), dtype=int)
for idx, dataset in enumerate(combine):
    for i in range(2):
        for j in range(3):
            sub = dataset[(dataset['Sex'] == i) & (dataset['Pclass'] == j+1)]['Age'].dropna()
            guess_ages[i, j] = int(sub.median())
    for i in range(2):
        for j in range(3):
            dataset.loc[(dataset['Age'].isnull()) & (dataset['Sex'] == i) & (dataset['Pclass'] == j+1), 'Age'] = guess_ages[i, j]
    dataset['Age'] = dataset['Age'].astype(int)

# Age bands
for dataset in combine:
    dataset.loc[dataset['Age'] <= 16, 'Age'] = 0
    dataset.loc[(dataset['Age'] > 16) & (dataset['Age'] <= 32), 'Age'] = 1
    dataset.loc[(dataset['Age'] > 32) & (dataset['Age'] <= 48), 'Age'] = 2
    dataset.loc[(dataset['Age'] > 48) & (dataset['Age'] <= 64), 'Age'] = 3
    dataset.loc[dataset['Age'] > 64, 'Age'] = 4
    dataset['Age'] = dataset['Age'].astype(int)

# IsAlone
for dataset in combine:
    dataset['FamilySize'] = dataset['SibSp'] + dataset['Parch'] + 1
    dataset['IsAlone'] = (dataset['FamilySize'] == 1).astype(int)
    dataset.drop(['Parch', 'SibSp', 'FamilySize'], axis=1, inplace=True)

# Age*Class
for dataset in combine:
    dataset['Age*Class'] = dataset['Age'] * dataset['Pclass']

# Embarked
freq_port = train_df['Embarked'].dropna().mode()[0]
for dataset in combine:
    dataset['Embarked'] = dataset['Embarked'].fillna(freq_port)
    dummies = pd.get_dummies(dataset['Embarked'])
    dataset[dummies.columns] = dummies
    dataset.drop('Embarked', axis=1, inplace=True)

# Fare
train_df['Fare'].fillna(train_df['Fare'].median(), inplace=True)
test_df['Fare'].fillna(train_df['Fare'].median(), inplace=True)
for dataset in combine:
    dataset['Fare'] = pd.qcut(dataset['Fare'], 4, labels=np.arange(4)).astype(int)

# Cabin and Ticket
train_df = train_df.drop(['Cabin', 'Ticket'], axis=1, errors='ignore')
test_df  = test_df.drop(['Cabin', 'Ticket'], axis=1, errors='ignore')
combine  = [train_df, test_df]

# Build submission template from test PassengerId
submission = pd.DataFrame({'PassengerId': test_df['PassengerId']})

print("Training shape:", train_df.shape, " | Test shape:", test_df.shape)
train_df.head(3)

Now we configure and train LightAutoML. Key parameters:

- `task = Task('binary')` - binary classification with AUC as the default metric.
- `timeout = 300` - LightAutoML will search for up to 5 minutes. Increase for a better-tuned result.
- `general_params={'use_algos': [...]}` - constrains which model families to try. Adding more families (e.g., `'cb'` for CatBoost) increases search time but can improve the final ensemble.

The `roles` dict tells LightAutoML which column is the target and which to ignore during training.

In [ ]:
# Define the prediction task
task = Task('binary')

# Roles: target column and columns to ignore
roles = {
    'target': 'Survived',
    'drop': ['PassengerId'],
}

# Initialize AutoML pipeline
# TODO: verify TabularUtilizedAutoML constructor params against current docs (retrieved 2026-08-25)
automl_lama = TabularUtilizedAutoML(
    task=task,
    timeout=TIMEOUT,
    cpu_limit=N_THREADS,
    general_params={'use_algos': [['linear_l2', 'lgb', 'lgb_tuned']]},
    reader_params={'n_jobs': N_THREADS}
)

# fit_predict returns out-of-fold predictions (not training predictions)
oof_pred = automl_lama.fit_predict(train_df, roles=roles)
print(f"OOF predictions shape: {oof_pred.shape}")
print(f"OOF AUC: {roc_auc_score(train_df['Survived'], oof_pred.data[:, 0]):.4f}")

**What is Out-of-Fold (OOF)?**

LightAutoML uses $k$-fold cross-validation internally. For each fold, it holds out a subset of training rows, trains on the remaining rows, and predicts the held-out subset. The OOF predictions are the collection of all held-out predictions across all folds.

This is more reliable than reporting training AUC (which the model can overfit to) or test AUC (which would leak information from the test set into model selection decisions). The OOF AUC estimates how well the model would generalize to truly new data.

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.3: GENERATING TEST PREDICTIONS

<br>

After training, we call `predict` on the test set to get probability scores. The output is a 2D array where `data[:, 0]` contains the probability that each passenger survived.

We save both the raw probability predictions (for later ensemble averaging) and the hard binary predictions (thresholded at 0.5, for a standalone submission).

In [ ]:
# Predict on test set
test_pred_lama = automl_lama.predict(test_df)
print(f"Test predictions shape: {test_pred_lama.shape}")
print(f"Sample probabilities (first 5): {test_pred_lama.data[:5, 0].round(3)}")

# Save probability predictions for ensemble use in Part 4
lama_proba = pd.Series(test_pred_lama.data[:, 0], index=test_df['PassengerId'], name='lama')

# Save hard binary predictions as a standalone Kaggle submission
submission['Survived'] = (test_pred_lama.data[:, 0] > 0.5).astype(int)
submission.to_csv('lightautoml_submission.csv', index=False)
print("Saved: lightautoml_submission.csv")
submission.head()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **LightAutoML reports an out-of-fold (OOF) AUC score during training. In 2-3 sentences, explain what 'out-of-fold' means in the context of k-fold cross-validation and why the OOF AUC gives a more reliable estimate of generalization performance than the training AUC.**

<br>

In [ ]:
# Written response question - answer in the markdown cell below


*ANSWER HERE*

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **AUTOML** WITH FLAML

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/flaml_logo.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

#### CONTENTS:

> [PART 3.1: Setup and Installation](#Part_3_1)<br>
> [PART 3.2: Training and Evaluation](#Part_3_2)<br>
> [PART 3.3: Inspecting the Best Model](#Part_3_3)<br>

<a id='Part_3_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.1: SETUP AND INSTALLATION

<br>

**FlaML** (Fast and Lightweight AutoML, from Microsoft Research) uses an economical hyperparameter search strategy that allocates more compute to configurations likely to perform well, and less to configurations that show poor early results. Unlike LightAutoML, FlaML does not build an internal ensemble - it finds the best single model configuration within the time budget.

This makes FlaML and LightAutoML complementary: LightAutoML tends to produce a blended multi-model prediction, while FlaML produces a single well-tuned model prediction. Combining them in Part 4 gives us some diversity.

**Sources Consulted:**
- FlaML: [microsoft.github.io/FLAML](https://microsoft.github.io/FLAML/docs/Use-Cases/Task-Oriented-AutoML/) - retrieved 2026-08-25
- FlaML API: [microsoft/FLAML on GitHub](https://github.com/microsoft/FLAML) - retrieved 2026-08-25

In [ ]:
# Install FlaML (skip if already installed)
# TODO: verify package version against current PyPI before pinning
!pip install flaml -q

In [ ]:
from flaml import AutoML

print("FlaML imported successfully")

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.2: TRAINING AND EVALUATION

<br>

FlaML uses a scikit-learn compatible API. The key configuration parameters:

- `time_budget` - total seconds for the search.
- `metric` - `'roc_auc'` instructs FlaML to optimize for AUC rather than accuracy. AUC is threshold-independent, which makes it more informative for imbalanced classification problems.
- `task` - `'classification'` for binary or multiclass problems.

FlaML's `fit` method returns the best found model. Calling `predict_proba` on the AutoML instance delegates to the best model's probability predictor.

In [ ]:
# Initialize and train FlaML AutoML
# TODO: verify AutoML constructor params against current flaml docs (retrieved 2026-08-25)
automl_flaml = AutoML()

automl_settings = {
    "time_budget": TIMEOUT,  # same time budget as LightAutoML for fair comparison
    "metric": 'roc_auc',
    "task": 'classification',
    "seed": RANDOM_STATE,
    "verbose": 1,
}

X_train = train_df.drop('Survived', axis=1)
y_train = train_df['Survived']

automl_flaml.fit(X_train=X_train, y_train=y_train, **automl_settings)

# Evaluate on training set (optimistic - use OOF or holdout for fair comparison)
flaml_train_proba = automl_flaml.predict_proba(X_train)[:, 1]
flaml_train_auc   = roc_auc_score(y_train, flaml_train_proba)
print(f"FlaML training AUC: {flaml_train_auc:.4f}")
print(f"Best model found: {automl_flaml.best_estimator}")

___

**Note:** FlaML reports training AUC rather than OOF AUC by default. The training AUC is optimistic (the model has seen these rows) and should not be directly compared to LightAutoML's OOF AUC - they measure different things. To compare fairly, you would either compute OOF predictions for FlaML manually using cross_val_predict, or compare both models on a held-out validation set.

___

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.3: INSPECTING THE BEST MODEL AND GENERATING PREDICTIONS

<br>

After training, FlaML exposes the best model configuration. Inspecting it tells us what algorithm and hyperparameters performed best within the budget - useful for understanding what the data prefers and for manually fine-tuning afterward.

In [ ]:
print("Best model estimator:", automl_flaml.best_estimator)
print("Best config:", automl_flaml.best_config)
print("Best loss (1 - AUC):", round(automl_flaml.best_loss, 4))

In [ ]:
# Generate test probability predictions
X_test = test_df.drop('PassengerId', axis=1, errors='ignore')
flaml_proba = pd.Series(
    automl_flaml.predict_proba(X_test)[:, 1],
    index=test_df['PassengerId'],
    name='flaml'
)

# Save standalone submission
submission['Survived'] = (flaml_proba.values > 0.5).astype(int)
submission.to_csv('flaml_submission.csv', index=False)
print("Saved: flaml_submission.csv")
print(f"Sample FlaML probabilities (first 5): {flaml_proba.values[:5].round(3)}")

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **Both LightAutoML and FlaML were given the same time budget (300 seconds) and the same data. Why might they select different best models even with identical constraints? Name at least two reasons, referencing what you know about how each framework searches.**

<br>

In [ ]:
# Written response question - answer in the markdown cell below


*ANSWER HERE*

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **COMBINING PREDICTIONS** AND THRESHOLD TUNING

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/ensemble_combination.png" align="center" width="40%" padding="10"><br>
    <br>
</div>

#### CONTENTS:

> [PART 4.1: Loading and Correlating Predictions](#Part_4_1)<br>
> [PART 4.2: Simple Averaging Ensemble](#Part_4_2)<br>
> [PART 4.3: Threshold Tuning](#Part_4_3)<br>
> [PART 4.4: Results Summary](#Part_4_4)<br>

<a id='Part_4_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.1: LOADING AND CORRELATING PREDICTIONS

<br>

We combine three probability prediction series:
1. LightAutoML (trained in Part 2)
2. FlaML (trained in Part 3)
3. H2O.ai (predictions from a separately-run H2O experiment, saved in `h2oai_experiment_macewube_test_predictions.csv`)

Before combining, we measure the pairwise correlations among the three prediction series. Recall from Part 1.3: low correlation between models is desirable for an ensemble. If two models produce nearly identical predictions, averaging them adds little beyond what either model achieves alone.

In [ ]:
# Load H2O predictions (generated from a separate H2O AutoML run)
# The H2O output file has columns: PassengerId, Survived.0 (P(not survive)), Survived.1 (P(survive))
h2o_raw = pd.read_csv('h2oai_experiment_macewube_test_predictions.csv')
h2o_proba = (h2o_raw
             .set_index('PassengerId')
             .rename(columns={'Survived.1': 'h2o'})['h2o'])

# Align all three series on PassengerId index
all_preds = pd.concat([lama_proba, flaml_proba, h2o_proba], axis=1)
all_preds.columns = ['lightautoml', 'flaml', 'h2o']

print("Prediction series shapes:", all_preds.shape)
print()
print("Pairwise correlations among model predictions:")
print(all_preds.corr().round(3))

High correlations (e.g., above 0.9) indicate that the models are making similar predictions. In that case, averaging them reduces variance only marginally - the ensemble behaves almost like any individual model. Lower correlations indicate genuine disagreement, and averaging has more to gain.

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.2: SIMPLE AVERAGING ENSEMBLE

<br>

Simple averaging takes the arithmetic mean of each model's survival probability for each passenger. No weighting by model quality, no meta-learner - just the mean. This is the most interpretable ensemble strategy and often performs within a few percentage points of more complex approaches like stacking.

The mean probability is then thresholded to produce a binary prediction: 0 or 1. The default threshold of 0.5 assumes equal cost for false positives and false negatives. Part 4.3 explores what happens when we shift this threshold.

In [ ]:
# Simple average ensemble
ensemble_proba = all_preds.mean(axis=1)

# Hard prediction at 0.5 threshold
ensemble_pred = (ensemble_proba > 0.5).astype(int)
ensemble_pred.name = 'Survived'

# Save ensemble submission
ensemble_submission = pd.DataFrame({
    'PassengerId': ensemble_proba.index,
    'Survived': ensemble_pred.values
})
ensemble_submission.to_csv('ensemble_avg_th050.csv', index=False)
print("Saved: ensemble_avg_th050.csv")
print(f"Predicted survivors: {ensemble_pred.sum()} / {len(ensemble_pred)}")

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.3: THRESHOLD TUNING

<br>

The threshold converts a continuous probability into a binary decision. The default of 0.5 maximizes accuracy on balanced datasets, but the Titanic training set is 62% non-survivors / 38% survivors - not severely imbalanced, but worth checking.

Lowering the threshold (e.g., to 0.45) makes the classifier more willing to predict survival, increasing recall at the cost of precision. Raising it (e.g., to 0.55) makes it more conservative. On a held-out validation set you would tune the threshold by optimizing the metric that matters for the downstream decision. Here we explore three thresholds to observe their effect on the predicted survivor count.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

thresholds = [0.45, 0.50, 0.55]
results = {}

for th in thresholds:
    preds = (ensemble_proba > th).astype(int)
    results[th] = {
        'survivors_predicted': int(preds.sum()),
        'non_survivors_predicted': int((preds == 0).sum()),
        'survival_rate': float(preds.mean())
    }
    preds.name = 'Survived'
    fname = f'ensemble_avg_th{int(th*100):03d}.csv'
    pd.DataFrame({'PassengerId': ensemble_proba.index, 'Survived': preds.values}).to_csv(fname, index=False)
    print(f"Threshold {th}: {results[th]['survivors_predicted']} predicted survivors ({results[th]['survival_rate']:.1%}) -> saved {fname}")

# Show how threshold shifts predicted survival rate
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, [results[t]['survival_rate'] for t in thresholds], marker='o', color='#003262')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Predicted Survival Rate')
ax.set_title('Predicted Survival Rate vs. Decision Threshold')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4_4'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.4: RESULTS SUMMARY

<br>

## **Results** on Kaggle Test Set

The table below summarizes the Kaggle leaderboard scores achieved by each submission in this notebook, along with the single-model baseline from a traditional sklearn pipeline for reference.

| Model | Kaggle Accuracy |
|---|---|
| Historical sklearn baseline (Logistic Regression / Random Forest) | 77.5% |
| H2O.ai AutoML | 77.9% |
| Simple average (LightAutoML + FlaML + H2O.ai), threshold 0.50 | 78.2% |

The ensemble outperforms every individual component, consistent with ensemble theory: each AutoML tool makes different errors, and the average cancels some of those errors.

**Key takeaways:**

1. The gain from ensembling is modest here (~0.3 percentage points) because the three models are highly correlated on this small dataset - they have learned similar patterns.
2. On larger datasets with more diverse feature spaces, the same simple averaging strategy often yields larger gains.
3. Threshold tuning on a held-out validation set (not the Kaggle test set) is the correct way to select a decision boundary. Using the test set for threshold selection would be data leakage.

**What to try next:**

- Add more diverse base models (e.g., a vanilla logistic regression or a k-nearest neighbor model) to increase ensemble diversity.
- Try stacking: train a meta-model on the OOF predictions of each base model.
- Experiment with weighted averaging, giving higher weight to the model with the best OOF AUC.

**Sources Consulted:**
- Kaggle Titanic historical results: [Kaggle Titanic Leaderboard](https://www.kaggle.com/c/titanic/leaderboard) - retrieved 2026-08-25
- AutoML comparison: [Kaggle Notebook: AutoML Libraries Comparison](https://www.kaggle.com/andreshg/automl-libraries-comparison)

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **The simple averaging ensemble achieves 78.2% accuracy while H2O alone achieves 77.9%. Using the correlation table computed in Part 4.1: explain in 2-3 sentences why adding the correlated LightAutoML and FlaML predictions still improved performance, and describe one change to the setup that would likely produce a larger improvement.**

<br>

In [ ]:
# Written response question - answer in the markdown cell below


*ANSWER HERE*

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 6px solid#003262;" />